# Sistem Pakar Diagnosis Kerusakan Hardware Laptop (Dempster-Shafer)
**Kelompok 2, Pengantar Komputasi Lunak.** Komang Indra Pradnya (2308561114) dan Jevan Bernard Kaloko (2308561078).

Notebook ini menjalankan alur **Gejala → Kerusakan → Solusi**:
1. membaca dataset CSV (18 kerusakan, 67 gejala, 26 rule, 71 relasi),
2. menggabungkan gejala yang dipilih dengan aturan kombinasi Dempster,
3. menampilkan kerusakan beserta interval [Belief, Plausibility],
4. menampilkan solusi.

**Catatan status yang jujur:**
- Data gejala dan nilai m berasal dari jurnal (Saragih dkk. 2018 sebagai basis, dilengkapi Arifin & Norhikmah 2024 dan Hilda dkk. 2023).
- **Solusi masih draf kelompok dan belum divalidasi** pakar atau dokumentasi servis.
- Aturan untuk **hasil seri** masih sementara: kerusakan yang Belief-nya sama ditampilkan sebagai kandidat setara.
- Notebook ini memverifikasi perhitungan terhadap data. Notebook ini belum menguji kecocokan diagnosis dengan kerusakan laptop yang sebenarnya.

**Urutan blok:**
| Bagian | Tujuan |
|---|---|
| 1. Membaca data | memuat berkas CSV ke memori, memeriksa nilai m, dan memeriksa konsistensi data |
| 2. Tampilkan dataset | memperlihatkan seluruh dataset dengan rapi: ringkasan sumber, kerusakan, gejala, rule |
| 3. Aturan kombinasi | fungsi inti Dempster-Shafer (gabung gejala, hitung Belief dan Plausibility) |
| 4. Diagnosis dan solusi | satu fungsi untuk memakai sistem: gejala masuk, kerusakan dan solusi keluar; ditambah tampilan langkah demi langkah |
| 5. Contoh dari laporan | angka yang sama dengan Bab IV laporan, termasuk contoh dengan hasil seri |
| 6. Verifikasi kasus uji | membuktikan perhitungan cocok dengan nilai yang diharapkan |
| 7. Cobalah sendiri | mencoba gejala pilihan sendiri |

**Cara memakai di Kaggle:** unggah keenam berkas CSV (kerusakan, gejala, relasi, rule, solusi, kasus_uji) sebagai satu *Dataset*, tambahkan dataset itu ke notebook lewat *Add Input*, lalu jalankan semua sel.
**Cara memakai di Google Colab:** unggah keenam berkas CSV ke panel *Files*, lalu jalankan semua sel. Lokasi data dikenali otomatis.

## 1. Membaca data
**Tujuan:** memuat CSV, menyiapkan kamus nama, nilai m, dan himpunan kerusakan tiap gejala, serta memastikan semua nilai m berada di antara 0 dan 1.

In [1]:
import os, glob, csv
from fractions import Fraction
from collections import defaultdict
import pandas as pd
pd.set_option("display.max_rows", None)

def cari_folder_data():
    pola = ["/kaggle/input/**/kerusakan.csv", "/content/**/kerusakan.csv", "/content/drive/MyDrive/**/kerusakan.csv"]
    for p in pola:
        hit = glob.glob(p, recursive=True)
        if hit:
            return os.path.dirname(hit[0])
    if os.path.exists("kerusakan.csv"):
        return "."
    raise FileNotFoundError(
        "kerusakan.csv tidak ditemukan. Di Kaggle: klik Add Input lalu tambahkan dataset CSV kelompok. "
        "Di Colab: unggah keenam berkas CSV ke panel Files.")

DATA = cari_folder_data()
def baca(nama):
    with open(os.path.join(DATA, nama + ".csv"), newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

kerusakan, gejala, relasi, rule = baca("kerusakan"), baca("gejala"), baca("relasi"), baca("rule")
solusi = {r["kode_kerusakan"]: r for r in baca("solusi")}
NAMA_K = {r["kode_kerusakan"]: r["nama_kerusakan"] for r in kerusakan}
NAMA_G = {r["kode_gejala"]: r["nama_gejala"] for r in gejala}
M_G = {r["kode_gejala"]: Fraction(r["nilai_m"]) for r in gejala}

# Himpunan kerusakan yang ditunjuk tiap gejala: diturunkan dari relasi (rule)
H = defaultdict(set)
for r in relasi:
    H[r["kode_gejala"]].add(r["kode_kerusakan"])
H = {g: frozenset(s) for g, s in H.items()}
THETA = frozenset(NAMA_K)          # ruang pembicaraan: seluruh kerusakan

print(f"Folder data : {DATA}")
print(f"Kerusakan {len(kerusakan)} | gejala {len(gejala)} | rule {len(rule)} | relasi {len(relasi)}")
assert all(0 < v < 1 for v in M_G.values()), "nilai m harus di antara 0 dan 1"

def label(A):
    """Tulis himpunan kerusakan, misalnya {K1, K2}; seluruh ruang pembicaraan ditulis θ."""
    return "θ" if A == THETA else "{" + ", ".join(sorted(A, key=lambda x: int(x[1:]))) + "}"


Folder data : /kaggle/input/datasets/mangin2005/dempster-shafer-kerusakan-laptop
Kerusakan 18 | gejala 67 | rule 26 | relasi 71


### Pengecekan konsistensi data
**Tujuan:** memastikan dataset tidak rusak sebelum dipakai. Bila ada yang salah (kode ganda, rule yang merujuk kode yang tidak ada, relasi yang tidak sesuai rule), notebook berhenti dengan pesan yang jelas, bukan menghasilkan angka yang keliru.

In [2]:
kode_k, kode_g = set(NAMA_K), set(NAMA_G)
assert len(kode_k) == len(kerusakan), "ada kode kerusakan ganda"
assert len(kode_g) == len(gejala), "ada kode gejala ganda"
assert len({r["no_rule"] for r in rule}) == len(rule), "ada nomor rule ganda"

pasangan_rule = set()
for r in rule:
    assert r["kerusakan"] in kode_k, f'{r["no_rule"]}: kerusakan {r["kerusakan"]} tidak ada di daftar kerusakan'
    for g in r["gejala"].split(";"):
        assert g in kode_g, f'{r["no_rule"]}: gejala {g} tidak ada di daftar gejala'
        pasangan_rule.add((g, r["kerusakan"]))
pasangan_relasi = {(r["kode_gejala"], r["kode_kerusakan"]) for r in relasi}
assert pasangan_rule == pasangan_relasi, "berkas relasi tidak sama dengan pasangan yang diturunkan dari rule"
assert all(g in H for g in kode_g), "ada gejala yang tidak dipakai rule mana pun"
assert set(solusi) == kode_k, "setiap kerusakan harus memiliki satu baris solusi"
assert all(h != THETA for h in H.values()), "ada gejala yang menunjuk seluruh kerusakan"

print("Pengecekan konsistensi lulus:")
print(f"  - {len(kerusakan)} kode kerusakan, {len(gejala)} kode gejala, dan {len(rule)} nomor rule semuanya unik")
print(f"  - semua rule merujuk kode yang ada; {len(pasangan_relasi)} pasangan relasi sama dengan yang diturunkan dari rule")
print("  - setiap gejala dipakai paling sedikit oleh satu rule, dan setiap kerusakan memiliki baris solusi")

Pengecekan konsistensi lulus:
  - 18 kode kerusakan, 67 kode gejala, dan 26 nomor rule semuanya unik
  - semua rule merujuk kode yang ada; 71 pasangan relasi sama dengan yang diturunkan dari rule
  - setiap gejala dipakai paling sedikit oleh satu rule, dan setiap kerusakan memiliki baris solusi


## 2. Tampilkan dataset
**Tujuan:** memperlihatkan seluruh data yang dipakai sistem dengan rapi, sebelum dihitung. Bagian ini hanya menampilkan data; tidak ada perhitungan.
Kolom *Sumber* menunjukkan jurnal asal tiap entri (Saragih dkk. 2018 sebagai basis; Arifin & Norhikmah 2024 dan Hilda dkk. 2023 sebagai pelengkap).

In [3]:
def sumber_dari(teks):
    return "Saragih" if "Saragih" in teks else "Arifin" if "Arifin" in teks else "Hilda"

def tampil(df, fmt=None):
    # tabel rata kiri, seluruh baris ditampilkan, tanpa nomor indeks
    display(df.style.format(fmt or {}).hide(axis="index")
              .set_properties(**{"text-align": "left", "white-space": "normal"})
              .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

# (a) ringkasan jumlah entri menurut sumber
ringkas = []
for nama, data in (("Kerusakan", kerusakan), ("Gejala", gejala), ("Rule", rule), ("Relasi gejala-kerusakan", relasi)):
    n = {"Saragih": 0, "Arifin": 0, "Hilda": 0}
    for r in data:
        n[sumber_dari(r["sumber"])] += 1
    ringkas.append((nama, n["Saragih"], n["Arifin"], n["Hilda"], sum(n.values())))
print("(a) Ringkasan dataset menurut sumber")
tampil(pd.DataFrame(ringkas, columns=["Jenis data", "Saragih", "Arifin", "Hilda", "Total"]))

# (b) daftar kerusakan beserta status solusinya
jml_gejala = defaultdict(int)
for r in relasi:
    jml_gejala[r["kode_kerusakan"]] += 1
print("\n(b) Daftar kerusakan")
tampil(pd.DataFrame([(r["kode_kerusakan"], r["nama_kerusakan"], sumber_dari(r["sumber"]), jml_gejala[r["kode_kerusakan"]],
                      solusi[r["kode_kerusakan"]]["tingkat"], solusi[r["kode_kerusakan"]]["status"]) for r in kerusakan],
                    columns=["Kode", "Kerusakan", "Sumber", "Jumlah gejala terkait", "Tingkat penanganan", "Status solusi"]))

(a) Ringkasan dataset menurut sumber


Jenis data,Saragih,Arifin,Hilda,Total
Kerusakan,15,2,1,18
Gejala,52,6,9,67
Rule,15,4,7,26
Relasi gejala-kerusakan,55,6,10,71



(b) Daftar kerusakan


Kode,Kerusakan,Sumber,Jumlah gejala terkait,Tingkat penanganan,Status solusi
K1,IC VGA,Saragih,3,KEDUANYA,DRAFT
K2,IC Power,Saragih,3,TEKNISI,DRAFT
K3,Inverter/gangguan pada kabel fleksibel laptop,Saragih,4,TEKNISI,DRAFT
K4,LCD,Saragih,5,KEDUANYA,DRAFT
K5,Keyboard laptop,Saragih,3,KEDUANYA,DRAFT
K6,TouchPad laptop,Saragih,5,KEDUANYA,DRAFT
K7,Harddisk laptop,Saragih,8,KEDUANYA,DRAFT
K8,Charger laptop,Saragih,4,KEDUANYA,DRAFT
K9,Sound Card,Saragih,4,KEDUANYA,DRAFT
K10,RAM,Saragih,3,KEDUANYA,DRAFT


In [4]:
# (c) daftar gejala: kode, bunyi gejala, nilai m, kerusakan yang ditunjuk
print("(c) Daftar gejala (67)")
tampil(pd.DataFrame([(r["kode_gejala"], r["nama_gejala"], float(Fraction(r["nilai_m"])), label(H[r["kode_gejala"]]), sumber_dari(r["sumber"]))
                     for r in gejala], columns=["Kode", "Gejala", "m", "Menunjuk", "Sumber"]), {"m": "{:.2f}"})

# (d) daftar rule: kombinasi gejala yang mengarah ke satu kerusakan
print("\n(d) Daftar rule (26)")
tampil(pd.DataFrame([(r["no_rule"], ", ".join(r["gejala"].split(";")), f'{r["kerusakan"]} {NAMA_K[r["kerusakan"]]}', sumber_dari(r["sumber"]))
                     for r in rule], columns=["Rule", "Gejala", "Kerusakan", "Sumber"]))

(c) Daftar gejala (67)


Kode,Gejala,m,Menunjuk,Sumber
G1,Suhu laptop meningkat secara drastis saat pertama kali dihidupkan,0.60,"{K1, K2}",Saragih
G2,Ukuran resolusi layar mengecil atau teks dan icon pada layar laptop membesar,0.50,{K1},Saragih
G3,Indikator lampu LED power menyala saat laptop dihidupkan tetapi gambar tidak muncul,0.70,{K1},Saragih
G4,Indikator lampu LED power tidak menyala saat laptop dihidupkan,0.70,{K2},Saragih
G5,Laptop tidak menampilkan gambar di layar / gelap,0.60,"{K3, K4, K10}",Saragih
G6,Cahaya pada layar laptop redup gelap namun menampilkan gambar,0.80,{K3},Saragih
G7,Layar kadang hidup mati dalam menampilkan gambar,0.50,{K3},Saragih
G8,Gambar yang ditampilkan bergetar / abnormal,0.60,{K3},Saragih
G9,Terdapat garis-garis pada layar/LCD laptop,0.80,{K4},Saragih
G10,"Terdapat titik-titik pada beberapa area layar/LCD laptop, bagian lainnya memiliki area terang",0.50,{K4},Saragih



(d) Daftar rule (26)


Rule,Gejala,Kerusakan,Sumber
R1,"G1, G2, G3",K1 IC VGA,Saragih
R2,"G1, G4",K2 IC Power,Saragih
R3,"G5, G6, G7, G8",K3 Inverter/gangguan pada kabel fleksibel laptop,Saragih
R4,"G5, G9, G10, G11, G12",K4 LCD,Saragih
R5,"G13, G14, G15",K5 Keyboard laptop,Saragih
R6,"G16, G17, G18, G19, G20",K6 TouchPad laptop,Saragih
R7,"G21, G22, G23, G24, G25, G26",K7 Harddisk laptop,Saragih
R8,"G27, G28, G29, G30",K8 Charger laptop,Saragih
R9,"G31, G32, G33",K9 Sound Card,Saragih
R10,"G5, G34, G35",K10 RAM,Saragih


## 3. Aturan kombinasi Dempster
**Tujuan:** menyediakan fungsi inti. Blok ini tidak mencetak apa-apa; ia hanya mendefinisikan fungsi yang dipakai blok sesudahnya.
Tiap gejala `g` memberi massa `m(g)` pada himpunan kerusakan yang ditunjuknya, dan `1 − m(g)` pada θ (ketidaktahuan).
Dua bukti digabungkan dengan tabel irisan: massa pada irisan yang sama dijumlahkan, massa pada irisan kosong adalah konflik **K**, lalu hasil dibagi (1 − K).
Perhitungan memakai pecahan eksak (`Fraction`) supaya tidak ada galat pembulatan.

In [5]:
def gabung(massa, g):
    """Gabungkan massa yang ada dengan gejala g. Mengembalikan (massa baru, konflik K, sel-sel tabel irisan)."""
    baru, K, sel = defaultdict(Fraction), Fraction(0), []
    for A, a in massa.items():
        for B, b in ((H[g], M_G[g]), (THETA, 1 - M_G[g])):
            C_ = A & B
            sel.append((label(A), label(B), label(C_) if C_ else "∅ (konflik)", a * b))
            if C_:
                baru[C_] += a * b
            else:
                K += a * b
    if K >= 1:
        raise ValueError("Konflik total: tidak ada massa tersisa")
    return {A: v / (1 - K) for A, v in baru.items()}, K, sel

def kombinasi(daftar_gejala):
    massa, langkah = {THETA: Fraction(1)}, []
    for g in daftar_gejala:
        massa, K, sel = gabung(massa, g)
        langkah.append((g, K, sel, dict(massa)))
    return massa, langkah

def bel_pl(massa, k):
    bel = massa.get(frozenset([k]), Fraction(0))
    pl = sum(v for A, v in massa.items() if k in A)
    return bel, pl

## 4. Diagnosis dan solusi
**Tujuan:** menyatukan semuanya menjadi satu fungsi `diagnosis([...])` yang dipanggil dengan daftar kode gejala.
- Peringkat diurutkan menurut Belief, lalu Plausibility.
- **Aturan sementara untuk seri:** bila beberapa kerusakan memiliki Belief tertinggi yang sama, semuanya ditampilkan sebagai *kandidat setara*.
- Bila tidak ada kerusakan tunggal dengan Belief positif, yang ditampilkan hanya himpunan kandidat.

In [6]:
def diagnosis(daftar_gejala, tampil_solusi=True):
    daftar_gejala = [g.strip().upper() for g in daftar_gejala]
    tak_dikenal = [g for g in daftar_gejala if g not in M_G]
    if tak_dikenal:
        raise ValueError(f"Kode gejala tidak ada dalam dataset: {tak_dikenal}")
    massa, langkah = kombinasi(daftar_gejala)

    print("Gejala dipilih:")
    for g in daftar_gejala:
        print(f"  {g}  m={float(M_G[g]):.2f}  menunjuk {label(H[g])}  | {NAMA_G[g]}")
    print("Konflik K tiap langkah:", [round(float(K), 4) for _, K, _, _ in langkah])

    baris = []
    for k in NAMA_K:
        bel, pl = bel_pl(massa, k)
        if pl > 0:
            baris.append((k, NAMA_K[k], bel, pl))
    baris.sort(key=lambda r: (-r[2], -r[3]))
    tabel = pd.DataFrame([(k, n, round(float(b), 4), round(float(p), 4)) for k, n, b, p in baris],
                         columns=["Kode", "Kerusakan", "Belief", "Plausibility"])
    print("\nHasil gabungan (massa):", {label(A): round(float(v), 4) for A, v in sorted(massa.items(), key=lambda kv: -kv[1])})
    display(tabel)

    tunggal = [r for r in baris if r[2] > 0]
    if not tunggal:
        print("HASIL: tidak ada kerusakan tunggal dengan Belief positif. Kandidat berupa himpunan:",
              [label(A) for A in massa if A != THETA])
        utama = []
    else:
        top = max(r[2] for r in tunggal)
        utama = [r for r in tunggal if r[2] == top]
        if len(utama) == 1:
            print(f"HASIL: {utama[0][1]} ({utama[0][0]}), Belief {float(utama[0][2]):.3f}, Plausibility {float(utama[0][3]):.3f}")
        else:
            print("HASIL: SERI (aturan sementara: kandidat setara) ->", ", ".join(f"{r[1]} ({r[0]})" for r in utama),
                  f"dengan Belief {float(top):.3f}. Disarankan menambah gejala.")
    if tampil_solusi:
        for r in utama:
            s = solusi.get(r[0])
            print(f"\nSOLUSI untuk {r[1]} [{s['status']}: {s['sumber_solusi']}]")
            print("  Tingkat penanganan:", s["tingkat"])
            for bagian in s["solusi"].split(" || "):
                judul, _, isi = bagian.partition(": ")
                print(f"  {judul}:")
                for i, langkah_ in enumerate(isi.split(" | "), 1):
                    print(f"    {i}. {langkah_}")
        if utama:
            print("\nPeringatan umum: matikan laptop dan lepas adaptor sebelum membuka atau membersihkan; bila masih bergaransi, jangan dibongkar sendiri.")
    return massa

### Tampilan langkah demi langkah
**Tujuan:** memperlihatkan setiap penggabungan secara rinci: tabel irisan, massa konflik K, dan hasil setelah normalisasi. Berguna untuk memeriksa angka dengan tangan atau untuk presentasi. Blok ini hanya mendefinisikan fungsi `langkah_demi_langkah([...])`; fungsi itu dipanggil pada Contoh 3 (bagian 5) dan dapat diaktifkan pada bagian 7 dengan `RINCI = True`.

In [7]:
def langkah_demi_langkah(daftar_gejala):
    """Tampilkan tiap penggabungan: tabel irisan, konflik K, dan hasil setelah normalisasi."""
    daftar_gejala = [g.strip().upper() for g in daftar_gejala]
    tak_dikenal = [g for g in daftar_gejala if g not in M_G]
    if tak_dikenal:
        raise ValueError(f"Kode gejala tidak ada dalam dataset: {tak_dikenal}")
    _, langkah = kombinasi(daftar_gejala)
    sebelumnya = {THETA: Fraction(1)}
    print("Mulai: belum ada bukti, seluruh massa pada θ  (m(θ) = 1)")
    for i, (g, K, sel, sesudah) in enumerate(langkah, 1):
        h = H[g]
        print(f"\n=== Langkah {i}: gabungkan {g} ({NAMA_G[g]}) ===")
        print(f"m({label(h)}) = {float(M_G[g]):.2f}   m(θ) = {float(1 - M_G[g]):.2f}")
        massa_lama = {label(A): a for A, a in sebelumnya.items()}
        baris, sebelum_norm = [], defaultdict(Fraction)
        for lab_a, lab_b, lab_c, v in sel:
            massa_b = M_G[g] if lab_b == label(h) else 1 - M_G[g]
            baris.append((f"{lab_a} = {float(massa_lama[lab_a]):.4f}", f"{lab_b} = {float(massa_b):.2f}", lab_c, round(float(v), 4)))
            if not lab_c.startswith("∅"):
                sebelum_norm[lab_c] += v
        display(pd.DataFrame(baris, columns=["Massa sebelumnya", f"Gejala {g}", "Irisan", "Hasil kali"]))
        print(f"Konflik K = {float(K):.4f}   penyebut (1 - K) = {float(1 - K):.4f}")
        print("Sebelum normalisasi:", {a: round(float(v), 4) for a, v in sebelum_norm.items()})
        print("Sesudah normalisasi:", {label(A): round(float(v), 4) for A, v in sorted(sesudah.items(), key=lambda kv: -kv[1])})
        sebelumnya = sesudah

## 5. Contoh yang sama dengan laporan (Bab IV)
**Tujuan:** memperlihatkan bahwa angka pada laporan dapat direproduksi: contoh 1 tanpa konflik, contoh 2 dengan konflik dan normalisasi.

In [8]:
print("=== Contoh 1: G1 dan G2 (tanpa konflik) ===")
_ = diagnosis(["G1", "G2"])

=== Contoh 1: G1 dan G2 (tanpa konflik) ===
Gejala dipilih:
  G1  m=0.60  menunjuk {K1, K2}  | Suhu laptop meningkat secara drastis saat pertama kali dihidupkan
  G2  m=0.50  menunjuk {K1}  | Ukuran resolusi layar mengecil atau teks dan icon pada layar laptop membesar
Konflik K tiap langkah: [0.0, 0.0]

Hasil gabungan (massa): {'{K1}': 0.5, '{K1, K2}': 0.3, 'θ': 0.2}


,Kode,Kerusakan,Belief,Plausibility
0,K1,IC VGA,0.5,1.0
1,K2,IC Power,0.0,0.5
2,K3,Inverter/gangguan pada kabel fleksibel laptop,0.0,0.2
3,K4,LCD,0.0,0.2
4,K5,Keyboard laptop,0.0,0.2
5,K6,TouchPad laptop,0.0,0.2
6,K7,Harddisk laptop,0.0,0.2
7,K8,Charger laptop,0.0,0.2
8,K9,Sound Card,0.0,0.2
9,K10,RAM,0.0,0.2


HASIL: IC VGA (K1), Belief 0.500, Plausibility 1.000

SOLUSI untuk IC VGA [DRAFT: Draf kelompok (pengetahuan umum perbaikan laptop), belum divalidasi pakar atau dokumentasi servis]
  Tingkat penanganan: KEDUANYA
  Dicoba sendiri:
    1. Sambungkan monitor atau TV eksternal; bila gambar tampil normal, masalah ada pada layar atau kabel layar, bukan pada VGA
    2. Perbarui atau instal ulang driver kartu grafis dari situs resmi produsen
    3. Matikan laptop, lepas adaptor (dan baterai bila bisa dilepas), tekan tombol power 30 detik, lalu nyalakan kembali
  Oleh teknisi:
    1. Pemeriksaan dan penggantian atau reball IC VGA oleh teknisi; jangan memanaskan papan sendiri

Peringatan umum: matikan laptop dan lepas adaptor sebelum membuka atau membersihkan; bila masih bergaransi, jangan dibongkar sendiri.


In [9]:
print("=== Contoh 2: G13 dan G49 (ada konflik, perlu normalisasi) ===")
massa, langkah = kombinasi(["G13", "G49"])
g, K, sel, _ = langkah[1]
display(pd.DataFrame([(a, b, c, round(float(v), 4)) for a, b, c, v in sel],
                     columns=["Massa sebelumnya", f"Gejala {g}", "Irisan", "Hasil kali"]))
print(f"Massa konflik K = {float(K):.2f}  ->  penyebut (1 - K) = {float(1 - K):.2f}")
_ = diagnosis(["G13", "G49"])

=== Contoh 2: G13 dan G49 (ada konflik, perlu normalisasi) ===


,Massa sebelumnya,Gejala G49,Irisan,Hasil kali
0,{K5},{K15},∅ (konflik),0.56
1,{K5},θ,{K5},0.14
2,θ,{K15},{K15},0.24
3,θ,θ,θ,0.06


Massa konflik K = 0.56  ->  penyebut (1 - K) = 0.44
Gejala dipilih:
  G13  m=0.70  menunjuk {K5}  | Ada sebagian/semua tombol keyboard yang tidak berfungsi
  G49  m=0.80  menunjuk {K15}  | Laptop tidak bisa membaca CD maupun DVD
Konflik K tiap langkah: [0.0, 0.56]

Hasil gabungan (massa): {'{K15}': 0.5455, '{K5}': 0.3182, 'θ': 0.1364}


,Kode,Kerusakan,Belief,Plausibility
0,K15,Pemutar CD/DVD,0.5455,0.6818
1,K5,Keyboard laptop,0.3182,0.4545
2,K1,IC VGA,0.0000,0.1364
3,K2,IC Power,0.0000,0.1364
4,K3,Inverter/gangguan pada kabel fleksibel laptop,0.0000,0.1364
5,K4,LCD,0.0000,0.1364
6,K6,TouchPad laptop,0.0000,0.1364
7,K7,Harddisk laptop,0.0000,0.1364
8,K8,Charger laptop,0.0000,0.1364
9,K9,Sound Card,0.0000,0.1364


HASIL: Pemutar CD/DVD (K15), Belief 0.545, Plausibility 0.682

SOLUSI untuk Pemutar CD/DVD [DRAFT: Draf kelompok (pengetahuan umum perbaikan laptop), belum divalidasi pakar atau dokumentasi servis]
  Tingkat penanganan: KEDUANYA
  Dicoba sendiri:
    1. Coba cakram lain yang bersih dan tidak tergores
    2. Bersihkan lensa dengan cakram pembersih, lalu periksa apakah drive terdeteksi oleh sistem
  Oleh teknisi:
    1. Penggantian drive CD/DVD oleh teknisi

Peringatan umum: matikan laptop dan lepas adaptor sebelum membuka atau membersihkan; bila masih bergaransi, jangan dibongkar sendiri.


### Contoh 3: kombinasi gejala yang menghasilkan seri tiga arah
**Tujuan:** memperlihatkan kasus dari laporan Bab IV (4.3.3). Gejala G9 (garis-garis pada layar), G6 (layar redup namun menampilkan gambar), dan G55 (laptop macet atau hang) memberi Belief yang sama untuk tiga kerusakan. Pada basis data Hilda saja, kombinasi gejala yang sama menghasilkan LCD sebagai diagnosis utama; perbedaannya bersumber dari gejala berstatus konflik pada pemetaan (lihat laporan).

In [10]:
print("=== Contoh 3: G9, G6, G55 (hasil seri) ===")
_ = diagnosis(["G9", "G6", "G55"])

=== Contoh 3: G9, G6, G55 (hasil seri) ===
Gejala dipilih:
  G9  m=0.80  menunjuk {K4}  | Terdapat garis-garis pada layar/LCD laptop
  G6  m=0.80  menunjuk {K3}  | Cahaya pada layar laptop redup gelap namun menampilkan gambar
  G55  m=0.80  menunjuk {K17}  | Laptop macet/hang
Konflik K tiap langkah: [0.0, 0.64, 0.7111]

Hasil gabungan (massa): {'{K4}': 0.3077, '{K3}': 0.3077, '{K17}': 0.3077, 'θ': 0.0769}


,Kode,Kerusakan,Belief,Plausibility
0,K3,Inverter/gangguan pada kabel fleksibel laptop,0.3077,0.3846
1,K4,LCD,0.3077,0.3846
2,K17,Processor Overheat,0.3077,0.3846
3,K1,IC VGA,0.0000,0.0769
4,K2,IC Power,0.0000,0.0769
5,K5,Keyboard laptop,0.0000,0.0769
6,K6,TouchPad laptop,0.0000,0.0769
7,K7,Harddisk laptop,0.0000,0.0769
8,K8,Charger laptop,0.0000,0.0769
9,K9,Sound Card,0.0000,0.0769


HASIL: SERI (aturan sementara: kandidat setara) -> Inverter/gangguan pada kabel fleksibel laptop (K3), LCD (K4), Processor Overheat (K17) dengan Belief 0.308. Disarankan menambah gejala.

SOLUSI untuk Inverter/gangguan pada kabel fleksibel laptop [DRAFT: Draf kelompok (pengetahuan umum perbaikan laptop), belum divalidasi pakar atau dokumentasi servis]
  Tingkat penanganan: TEKNISI
  Dicoba sendiri:
    1. Sambungkan monitor eksternal untuk memastikan laptop masih menghasilkan gambar
    2. Sorot layar dengan senter dari samping; bila gambar samar terlihat, kemungkinan lampu latar atau inverter bermasalah
    3. Hindari membuka dan menutup layar berulang kali sampai diperiksa
  Oleh teknisi:
    1. Penggantian kabel fleksibel atau inverter oleh teknisi

SOLUSI untuk LCD [DRAFT: Draf kelompok (pengetahuan umum perbaikan laptop), belum divalidasi pakar atau dokumentasi servis]
  Tingkat penanganan: KEDUANYA
  Dicoba sendiri:
    1. Sambungkan monitor eksternal; bila tampil normal, kerusak

**Langkah demi langkah untuk Contoh 3.** Tabel irisan di bawah menunjukkan mengapa konflik membesar pada langkah kedua dan ketiga (K = 0,64 lalu 0,7111) dan mengapa tiga kerusakan berakhir dengan massa yang sama.

In [11]:
langkah_demi_langkah(["G9", "G6", "G55"])

Mulai: belum ada bukti, seluruh massa pada θ  (m(θ) = 1)

=== Langkah 1: gabungkan G9 (Terdapat garis-garis pada layar/LCD laptop) ===
m({K4}) = 0.80   m(θ) = 0.20


,Massa sebelumnya,Gejala G9,Irisan,Hasil kali
0,θ = 1.0000,{K4} = 0.80,{K4},0.8
1,θ = 1.0000,θ = 0.20,θ,0.2


Konflik K = 0.0000   penyebut (1 - K) = 1.0000
Sebelum normalisasi: {'{K4}': 0.8, 'θ': 0.2}
Sesudah normalisasi: {'{K4}': 0.8, 'θ': 0.2}

=== Langkah 2: gabungkan G6 (Cahaya pada layar laptop redup gelap namun menampilkan gambar) ===
m({K3}) = 0.80   m(θ) = 0.20


,Massa sebelumnya,Gejala G6,Irisan,Hasil kali
0,{K4} = 0.8000,{K3} = 0.80,∅ (konflik),0.64
1,{K4} = 0.8000,θ = 0.20,{K4},0.16
2,θ = 0.2000,{K3} = 0.80,{K3},0.16
3,θ = 0.2000,θ = 0.20,θ,0.04


Konflik K = 0.6400   penyebut (1 - K) = 0.3600
Sebelum normalisasi: {'{K4}': 0.16, '{K3}': 0.16, 'θ': 0.04}
Sesudah normalisasi: {'{K4}': 0.4444, '{K3}': 0.4444, 'θ': 0.1111}

=== Langkah 3: gabungkan G55 (Laptop macet/hang) ===
m({K17}) = 0.80   m(θ) = 0.20


,Massa sebelumnya,Gejala G55,Irisan,Hasil kali
0,{K4} = 0.4444,{K17} = 0.80,∅ (konflik),0.3556
1,{K4} = 0.4444,θ = 0.20,{K4},0.0889
2,{K3} = 0.4444,{K17} = 0.80,∅ (konflik),0.3556
3,{K3} = 0.4444,θ = 0.20,{K3},0.0889
4,θ = 0.1111,{K17} = 0.80,{K17},0.0889
5,θ = 0.1111,θ = 0.20,θ,0.0222


Konflik K = 0.7111   penyebut (1 - K) = 0.2889
Sebelum normalisasi: {'{K4}': 0.0889, '{K3}': 0.0889, '{K17}': 0.0889, 'θ': 0.0222}
Sesudah normalisasi: {'{K4}': 0.3077, '{K3}': 0.3077, '{K17}': 0.3077, 'θ': 0.0769}


## 6. Verifikasi terhadap kasus uji
**Tujuan:** bukti bahwa perhitungan program cocok dengan nilai yang diharapkan.
Enam kasus uji pada `kasus_uji.csv` dihitung ulang dan dibandingkan dengan massa yang diharapkan (toleransi pembulatan dua desimal pada nilai yang ditulis desimal).

In [12]:
def urai(teks):
    out = {}
    for p in teks.split(";"):
        k, v = p.split("=")
        A = THETA if k == "THETA" else frozenset(k.split("|"))
        out[A] = Fraction(v)
    return out

hasil = []
for r in baca("kasus_uji"):
    gs = r["gejala_dipilih"].split(";")
    massa, _ = kombinasi(gs)
    harap = urai(r["massa_diharapkan"])
    cocok = set(massa) == set(harap) and all(abs(massa[A] - harap[A]) < Fraction(1, 200) for A in harap)
    hasil.append((r["id_kasus"], ", ".join(gs), r["diagnosis_utama"], "cocok" if cocok else "TIDAK COCOK"))
display(pd.DataFrame(hasil, columns=["Kasus", "Gejala", "Diagnosis di CSV", "Hasil"]))
assert all(h[3] == "cocok" for h in hasil), "ada kasus uji yang tidak cocok"
print("Semua kasus uji cocok.")

,Kasus,Gejala,Diagnosis di CSV,Hasil
0,KU01,"G1, G2",K1,cocok
1,KU02,"G1, G2, G3",K1,cocok
2,KU03,G9,K4,cocok
3,KU04,"G1, G5",TIDAK_TUNGGAL,cocok
4,KU05,"G13, G49",K15,cocok
5,KU06,"G9, G49",SERI,cocok


Semua kasus uji cocok.


## 7. Percobaan
**Tujuan:** menjalankan diagnosis dengan gejala pilihan sendiri. Isi daftar kode gejala pada sel di bawah (kodenya ada pada tabel gejala di bagian 2), lalu jalankan.

In [13]:
GEJALA_SAYA = ["G9", "G49"]      
RINCI = True
_ = diagnosis(GEJALA_SAYA)
if RINCI:
    langkah_demi_langkah(GEJALA_SAYA)

Gejala dipilih:
  G9  m=0.80  menunjuk {K4}  | Terdapat garis-garis pada layar/LCD laptop
  G49  m=0.80  menunjuk {K15}  | Laptop tidak bisa membaca CD maupun DVD
Konflik K tiap langkah: [0.0, 0.64]

Hasil gabungan (massa): {'{K4}': 0.4444, '{K15}': 0.4444, 'θ': 0.1111}


,Kode,Kerusakan,Belief,Plausibility
0,K4,LCD,0.4444,0.5556
1,K15,Pemutar CD/DVD,0.4444,0.5556
2,K1,IC VGA,0.0000,0.1111
3,K2,IC Power,0.0000,0.1111
4,K3,Inverter/gangguan pada kabel fleksibel laptop,0.0000,0.1111
5,K5,Keyboard laptop,0.0000,0.1111
6,K6,TouchPad laptop,0.0000,0.1111
7,K7,Harddisk laptop,0.0000,0.1111
8,K8,Charger laptop,0.0000,0.1111
9,K9,Sound Card,0.0000,0.1111


HASIL: SERI (aturan sementara: kandidat setara) -> LCD (K4), Pemutar CD/DVD (K15) dengan Belief 0.444. Disarankan menambah gejala.

SOLUSI untuk LCD [DRAFT: Draf kelompok (pengetahuan umum perbaikan laptop), belum divalidasi pakar atau dokumentasi servis]
  Tingkat penanganan: KEDUANYA
  Dicoba sendiri:
    1. Sambungkan monitor eksternal; bila tampil normal, kerusakan ada pada panel layar
    2. Atur ulang resolusi dan refresh rate, lalu perbarui driver grafis
    3. Jangan menekan atau membersihkan layar dengan cairan secara berlebihan
  Oleh teknisi:
    1. Penggantian panel LCD atau kabel layar oleh teknisi

SOLUSI untuk Pemutar CD/DVD [DRAFT: Draf kelompok (pengetahuan umum perbaikan laptop), belum divalidasi pakar atau dokumentasi servis]
  Tingkat penanganan: KEDUANYA
  Dicoba sendiri:
    1. Coba cakram lain yang bersih dan tidak tergores
    2. Bersihkan lensa dengan cakram pembersih, lalu periksa apakah drive terdeteksi oleh sistem
  Oleh teknisi:
    1. Penggantian drive CD/

,Massa sebelumnya,Gejala G9,Irisan,Hasil kali
0,θ = 1.0000,{K4} = 0.80,{K4},0.8
1,θ = 1.0000,θ = 0.20,θ,0.2


Konflik K = 0.0000   penyebut (1 - K) = 1.0000
Sebelum normalisasi: {'{K4}': 0.8, 'θ': 0.2}
Sesudah normalisasi: {'{K4}': 0.8, 'θ': 0.2}

=== Langkah 2: gabungkan G49 (Laptop tidak bisa membaca CD maupun DVD) ===
m({K15}) = 0.80   m(θ) = 0.20


,Massa sebelumnya,Gejala G49,Irisan,Hasil kali
0,{K4} = 0.8000,{K15} = 0.80,∅ (konflik),0.64
1,{K4} = 0.8000,θ = 0.20,{K4},0.16
2,θ = 0.2000,{K15} = 0.80,{K15},0.16
3,θ = 0.2000,θ = 0.20,θ,0.04


Konflik K = 0.6400   penyebut (1 - K) = 0.3600
Sebelum normalisasi: {'{K4}': 0.16, '{K15}': 0.16, 'θ': 0.04}
Sesudah normalisasi: {'{K4}': 0.4444, '{K15}': 0.4444, 'θ': 0.1111}


### Bila hasilnya seri
Pada contoh di atas, G9 (garis-garis pada layar) dan G49 (tidak bisa membaca CD/DVD) memberi Belief yang sama untuk LCD dan CD/DVD, sehingga sistem menampilkan keduanya sebagai kandidat setara. Menambah gejala yang relevan biasanya memecahkan seri. Aturan keputusan final masih menunggu arahan dosen.